In [7]:
import os # 환경 변수 설정에 사용됨
import re # 정규 표현식 처리 모듈임. ReAct에이전트의 출력에서 Action input, Final Answer 등을 추출하기 위해 사용함.
import io # 문자열을 파일 처럼 다룰수 있게 해주는 모듈임. StringIO를 사용해 에이전트의 실행 로그를 메모리상에서 캡처함.
import requests # HTTP 요청을 보내는 라이브러리임. PDF 파일을 url에서 다운로드 받을 때 사용함.
from dotenv import load_dotenv # 환경 변수 외부 설정에 사용됨.
from typing import Dict, List, Optional, Tuple # 함수의 매개변수나 반환값에 타입 힌트를 제공하는 모듈임. 코드 가독성을 높임.
from contextlib import redirect_stdout #컨텍스트 관리 유틸리티를 제공하는 모듈임. redirect_stdout를 사용해 에이전트 실행 중 출력을 다른 곳으로 보낼 수 있음.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader # langchain_community 는 커뮤니티에서 개발한 랭체인 확장 기능들을 제공함.
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool #langchain_core는 랭체인의 핵심 기능. create_retriever_tool은 검색기를 에이전트가 사용할 수있는 도구 형태로 변환.
from langchain_core.prompts import PromptTemplate # PromptTemplate는 프롬프트를 동적으로 생성하는 데 사용함.

import gradio as gr # 머신 러닝 모델을 위한 웹 인터페이스를 쉽고 빠르게 구축할 수있는 라이브러리임. 코드 몇 줄로 전문적인 채팅 데모 사이트를 만들 수 있음.

#os.environ["OPEN_API_KEY"]= ""
load_dotenv()

# 실습 데이터 다운로드.
"""
urls = [
    "https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/ict_japan_2024.pdf",
	"https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/ict_usa_2024.pdf",
	"https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/blockchain_usa_2025.pdf"
]
"""

# 각 파일을 다운로드.
"""
for url in urls :
    filename = url.split("/")[-1] # URL에서 파일명 추출
    response = requests.get(url)

    with open(filename, "wb") as f:  # with문은 "시작할때 이거 해두고, 끝날 때 무슨일이 있더라도 정리해줘" 를 보장해 주는 스마트한 문법임.
        f.write(response.content)

    print(f"{filename} 다운로드 완료")    
"""

# 임베딩 설정.
embd = OpenAIEmbeddings()

def create_pdf_retriever(
        pdf_path: str,                      # PDF 파일 경로
        persist_directory: str,             # 벡터 스토어 저장 경로(persist : 지속)
        embedding_model: OpenAIEmbeddings,  # OpenAIEmbeddings 임베딩 모듈
        chunk_size: int=512,                # 청크 크기 기본값 : 512
        chunk_overlap: int=0                # 청크 오버랩 크기 기본값 : 0
) -> Chroma.as_retriever:                                                   # 타입 힌트(Type Hint) : 파이썬은 원래 함수의 타입을 미리 정하지 않는 동적 타이핑 언어임
                                                                            # 하지만 코드의 가독성을 높이기 위해 def 함수이름(...) -> 반환타입: 의 형태로 명시할 수 있음.
    # PDF 파일 로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # chunking
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어로 적재
    vectorstore = Chroma.from_documents(
        persist_directory = persist_directory,
        documents = doc_splits,
        embedding = embedding_model
    )

    return vectorstore.as_retriever() # retriever 객체는 자연어 질의에 대해 관련성 높은 PDF 내용을 검색할 수있는 검색기임.

# 주제별로 독립적인 벡터 검색기를 생성함.
# 일본 ICT 정책 데이터베이스 생성.
retriever_ict_japan = create_pdf_retriever(
    pdf_path = "ict_japan_2024.pdf",
    persist_directory = "db_ict_policy_japan_2024",
    embedding_model = embd
)

# 미국 ICT 정책 데이터베이스 생성.
retriever_ict_usa = create_pdf_retriever(
    pdf_path = "ict_usa_2024.pdf",
    persist_directory = "db_ict_policy_usa_2024",
    embedding_model = embd
)

# 미국 블록체인 동향 데이터베이스 생성
retriever_blockchain_usa  = create_pdf_retriever(
    pdf_path = "blockchain_usa_2025.pdf",
    persist_directory = "db_blockchain_usa_2025",
    embedding_model = embd
)

# create_pdf_retriever 함수를 사용해 각각의 벡터 데이터베이스 검색기를 생성함.
# persist_directory 에 db_ict_policy_japan_2024, db_ict_policy_usa_2024, db_blockchain_usa_2025 라는 서로 다른 디렉토리 경로를 지정하면 
# 세개의 벡터 데이터베이스가 서로 영향을 주지 않고, 독립적으로 저장되고 관리함.

# 만약 같은 경로를 사용했다면 두번째 벡터 데이터베이스를 처리할 때 첫번째 벡터 데이터베이스를 덮어쓰거나 혼합될 수 있기 때문에 이렇게 경로를 분리하는 것이 중요함.

# 결론적으로 일본 ICT 정책을 다루는 ict_japan_2024.pdf 파일을 열어 db_ict_policy_japan_2024 디렉터리에 벡터 데이터베이스를 생성함.
# 마찬가지로 미국 ICT 정책을 다루는 ict_usa_2024.pdf 파일을 열어 db_ict_policy_usa_2024 디렉터리에,
# 미국 블록체인 동향에 대해 다루는 blockchain_usa_2025.pdf 파일을 db_blockchain_usa_2025 디렉터리에 벡터 데이터 베이스로 저장해 세개의 검색기를 만들었음.

# 이제 이렇게 만들어진 3개의 retriever 객체를 create_retriever_tool 함수를 통해 ReAct 에이전트가 사용할 수 있는 검색 도구 형태로 변환해 봄.
ict_japan_engine = create_retriever_tool(
    retriever = retriever_ict_japan,
    name = "japan_ict_trend_searcher",
    description = "일본의 ICT 산업의 시장 동향 정보를 제공합니다."
)

ict_usa_engine = create_retriever_tool(
    retriever = retriever_ict_usa,
    name = "usa_ict_trend_searcher",
    description = "미국의 ICT 산업의 시장 동향 정보를 제공합니다."
)

blockchain_usa_engine = create_retriever_tool(
    retriever = retriever_blockchain_usa,
    name = "usa_blockchain_trend_searcher",
    description = "미국의 블록체인 산업의 동향 정보를 제공합니다."
)

tools = [ict_japan_engine, ict_usa_engine, blockchain_usa_engine]
tool_map: Dict[str, object] = {t.name: t for t in tools}

# create_retriever_tool 함수는 retriever 객체를 ReAct 에이전트가 활용할 수 있는 형태의 도구로 변환하는 함수임.
# 앞에서 생성한 retriever_ict_japan 등의 객체를 바탕으로 도구의 이름(name)과 설명(description)을 추가해 ReAct 메이전트가 활용할 수 있는 도구로 만듬.

# 이때 도구를 생성할때 description 에는 각 검색기의 상세한 용도를 작성해야 함.
# 예를 들어, 일본 ICT 검색기의 경우, '일본의 ICT 시장 동향 정보를 제공합니다.'라는 설명을 작성했음.
# 이후 ReAct 에이전트가 동작할때 에이전트는 여기서 적힌 설명을 보고 사용자의 질문에 따라 도구를 선택함.
# 즉, 에이전트는 사용자가 미국의 ICT 산업과 관련된 질문을 하면 usa_ict라는 이름의 도구를 사용하고 일본의 ICT 산업과 관련된 질문을 하면 japan_ict 라는 이름의 도구를 사용하게 됨.
# 그리고 세개의 주제와 상관이 없는 질문에는 어떠한 도구도 사용하지 않고 답변을 하게 함.
# 이는 에이전트가 오로지 도구의 description을 보고 판단함. 따라서 description에 도구에 대해 설명이 제대로 적혀 있지 않다면 에이전트는 상황에 맞는 도구 선택을 제대로 할수 없으므로
# 상세한 설명을 기재해야 함.

# 생성된 세걔의 도구인 ict_japan_engine , ict_usa_engine, blockchain_usa_engine 을 tool 리스트에 담음.
# 이 tools 리스트를 뒤에 코드에서 ReAct 에이전트에 전달하고, 에이전트는 이 도구들의 description 을 통해 각 질문에 가장 적합한 도구를 선택하고 정확한 검색을 수행할 수 있게 함.

# tool_map 은 도구이 이름을 키(key), 도구 객체를 값(value)으로 답는 딕셔너리임.
# 이렇게 만들어 두면 에이전트가 "japan_ict_trend_searcher"라는 도구를 사용하겠다고 결정했을때 tool_map["japan_ict_trend_searcher"]로 해당 도구를 즉시 찾아서 실행할 수 있음.

# 이제 ReAct 에이전트가 사용할 프롬프트 템플릿을 설정하겠음.
# ReAct 에이전트는 정해진 형식에 따라 사고하고 행동해야 하므로 이 형식을 명확하게 지시하는 프롬프트가 필요함.
react_template = '''
다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:
{tools}

다음 형식을 사용하세요:
Question: 답변을 해야하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
...(이 Thought/Action/Action Input/Observation 의 과정이 N번 반복할 수 있습니다.)
Thought: 이제 최종 답변을 알겠습니다.
Final Answer: 원랙 입력된 질문에 대한 최종 답변

## 추가적인 주의사항
- 반드시 [Thought->Action->Action Input format] 이 사이클의 순서를 준수하십시오. 항상 Action 전에는 Thought 가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것이 중요합니다.
- 정보가 취합 되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question : {input}
{agent_scratchpad} 
'''
# scratchpad :고속 작업용 보조기억장치
# 실제 상황에서 {input} 부분에는 사용자가 입력한 질문이 그대로 들어감. 예를 들어, '일본의 AI 정책에 대해 알려줘'라고 질문하면 이 텍스트가 Question: 뒤에 입력됨.
# {agent_scratchpad} 부분은 에이전트가 사고 과정에 해당하는 ReAct 사이클이 지속적으로 누적되는 공간임.
# 처음에는 비어 있지만 에이전트가 실행되면서 Thought, Action, Action Input, Observation 이라는 react 사이클이 Final Answer 단계에 이르기까지 계속 누적되어 쌓임.
prompt = PromptTemplate.from_template(react_template)

# 이제 프롬프트 템플릿에 실제 값들을 채워 넣는 함수들을 구현하겠음.
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []

    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description", "")
        lines.append(f"{t.name}: {desc}")

    return "\n".join(lines), ", ".join(names)

def _render_prompt(user_input: str, scratchpad: str) -> str:
    tools_str, tool_names = _format_tools_for_prompt(tools)

    return prompt.format(
        tools= tools_str,
        tool_names = tool_names,
        input = user_input,
        agent_scratchpad = scratchpad
    )

# _format_tools_for_prompt 함수는 도구 리스트를 프롬프트에 삽입할 형태로 반환함.
# 이 함수는 두개의 빈 리스트인 lines와 names 를 초기화한 후 도구 리스트를 순회하면서 각 도구의 이름을 names 리스트에 추가하고 "도구이름":"설명" 형태의 문자열을 lines 리스트에 추가함.
# 최종적으로 lines는 줄바꿈으로 연결하고 names 는 쉼표로 연결해 두개의 문자열을 반환함.

# 예를 들어, lines는 "japan_ict_trend_searcher : 일본의 ICT 산업의 시장동향 정보를 제공합니다. \n..."
# names 는 "japan_ict_trend_searcher, usa_ict_trend_searcher, usa_blockchain_trend_searcher" 형태가 됨.

# _render_prompt 함수는 사용자 입력을 에이전트의 사고 과정(scratcher)을 받아 완성된 프롬프트를 생성함.
# 먼저 _format_tools_for_prompt 함수를 호출해 도구 정보를 문자열로 반환함. 그런 다음 prompt.format() 메서드로 템플릿의 {tools}, {tool_name}, {input}, {agent_scratchpad}에 실제값을 채움
# 이렇게 완성된 프롬프트가 LLM에게 전달되어 에이전트의 다음 행동을 결정하게 됨.

# 이제 LLM 초기화 하고 에이전트의 출력을 파싱하는 로직을 구현하겠음.
llm = ChatOpenAI(model="gpt-5.6", temperature=0)

# llm 변수에 GPT-5.6 모델로 초기화함. temperature=0 으로 설정하는 이유는 모델의 출력을 최대한 일관되게 만들기 위해서임.
# temperature는 LLM이 답변을 생성할 때, 창의성과 무작위성을 조절하는 매개변수임. 값이 0에 가까울수록 같은 답변을 생성하고, 1에 가까울소록 창의적인 답변을 생성함.
# ReAct 에이전트는 도구 선택과 논리적 추론이 중요하므로 일관된 판단을 위해 temperature를 0 으로 설정함.

# 다음으로 LLM의 출력에서 Action, Action Input, Final Answer 를 추출하기 위한 정규 표현식 패턴과 파싱 함수를 정의함.
ACTION_RE = re.compile(r"^Action\s*:\s*(?P<tool>.+?)\s*$", re.MULTILINE)
ACTION_INPUT_RE = re.compile(r"^Action Input\s*:\s*(?P<input>.+?)\s*$", re.MULTILINE)
FINAL_ANSWER_RE = re.compile(r"^Final Answer\s*:\s*(?P<final>[\s\S]+)$", re.IGNORECASE)

def _parse_action_and_input(text: str) -> Tuple[Optional[str], Optional[str]]:
    m_final = FINAL_ANSWER_RE.search(text)

    if m_final :
        return "__FINAL__", m_final.group("final").strip()

    m_act = ACTION_RE.search(text)
    m_in = ACTION_INPUT_RE.search(text)

    if m_act and m_in:
        return m_act.group("tool").strip(), m_in.group("input").strip()

    return None, None

def _observation_to_text(observation_obj) -> str:
    if isinstance(observation_obj, list):
        # Document 리스트를 알 수 있음.
        def doc_to_str(d):
            try:
                meta = getattr(d, "metadata, {}") or {}
                src = meta.get("source") or meta.get("file_path") or ""
                txt = getattr(d, "page_content", "")

                if len(txt) > 500:
                    txt = txt[:500] + "..."
                return f"[source={str}] {txt}"

            except Exception:
                return str(d)

        return "\n".join(doc_to_str(d) for d in observation_obj[:5])

    return str(observation_obj)

# 세개의 정규 표현식 패턴을 정의함.
# ACTION_RE 는 "Action: 도구이름" 형태의 텍스트에서 도구 이름을 추출하는 패턴임.
# ^Action\s*:\s* 는 줄의 시작에서 "Action:" 을 찾되 콜론 앞뒤의 공백을 허용하고, (?P<tool>.+?)는 도구이름을 tool 이라는 이름의 그룹으로 캡처함.
# re.MULTILINE 플래그는 ^가 각 줄의 시작을 의미하도록 함.

# ACTION_INPUT_RE 는 동일한 방식으로 "Action Input: 입력값" 형태에서 입력값을 추출함.
# FINAL_ANSWER_RE 는 "Final Answer:" 뒤에 오는 모든 텍스트를 최종 답변으로 추출하며 re.IGNORECASE 플래그로 대소문자를 구분하지 않음.

# _parse_action_and_input 함수는 LLM의 출력 테스트를 받아 에이전트가 어떤 행동을 취하려는지 파싱함.
# 먼저 Final Answer 가 있는지 확인하고, 있다면 "FINAL"이라는 특별한 문자열과 함께 최종 답변을 반환함.
# "FINAL"은 에이전트가 사고를 마치고 최종 답변을 생성했을을 나타냄. Final Answer 가 없다면 Action 과 Action Input 을 찾아서 반환함.
# 둘다 찾지 못하면 None, None 을 반환하여 파싱에 실패했음을 알림.

# 도구를 실행하면 다양한 형태의 결과가 반환함. 벡터 검색기의 경우 Document 객체들의 리스트가 반환되는데, 이를 에이전트가 이해할 수 있는 텍스트 형태로 변환하는 함수가 필요함.

# _observation_obj_to_text 함수는 도구 실행 결과를 문자열로 반환함.
# 먼저 결과가 리스트 인지 확인함. 리스트인 경우 Document 객체일 가능성이 높으므로 각 Document 를 문자열로 변환하는 내부 함수인 doc_to_str 을 정의함.
# 이 내부 함수는 Document의 metadata에서 출처 정보를 추출하고, page_content에서 텍스트 내용을 가져옴.
# 텍스트가 500자를 초과하면 잘라서 "..."를 붙여 프롬프트가 지나치게 길어지는 것을 방지함.
# 최종적으로 "[source=파일경로] 텍스트내용" 형태로 반환함.
# 리스트의 처음 5개 문서만 처리하여 결과를 적절한 길이로 유지함. 결과가 리스트가 아닌 경우에는 단순히 문자열로 변환하여 반환함.

# 이제 ReAct 에이전트의 핵심인 실행 루프를 구현하겠음.
# 이 함수는 사용자의 질문을 받아 Thought -> Action -> Action Input -> Observation 사이클을 반복하여 최종 답변을 생성함.

def run_react(user_input: str, max_iters: int = 8) -> Dict[str, str]:
    scratchpad = ""

    for _ in range(max_iters):
        rendered = _render_prompt(user_input, scratchpad)
        resp = llm.invoke(rendered)
        text = resp.content if hasattr(resp, "content") else str(resp)  # hasattr(object, "속성이름") : 객체에 특정 속성/함수가 있는지 여부 확인.

        tool, action_input = _parse_action_and_input(text)

        if tool is None:
            hint = "\n[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'와 'Action Input:'을 한줄씩 제공하십시오.\n"
            scratchpad += f"{text}\n{hint}"
            continue

        if tool == "__FINAL__":
            final_answer = action_input
            return {"output": final_answer, "log": scratchpad + "\n" + text}

        if tool not in tool_map:
            observation = f"[에러] 존재하지 않는 도구입니다: {tool}"
            scratchpad += f"{text}\nobservation: {observation}\n"
            continue

        try:
            observation_obj = tool_map[tool].invoke(action_input)
            observation = _observation_to_text(observation_obj)
            scratchpad += f"{text}\nobservation: {observation}\n"

        except Exception as e:
            scratchpad += f"{text}\nobservation: [도구실행 오류] {e}\n"

    return {
        "output": "반복 한도를 초과합니다. 질문을 더 구체화해 주세요.",
        "log": scratchpad,
    }   

# run_react 함수는 ReAct 에이전트의 핵심 실행 함수임.
# user_input 으로 사용자의 질문을 받고, max_iters 로 최대 반복 횟수를 설정함.
# 기본값은 8회로, 대부분의 질문이 8번 이내의 사이클로 해결된다고 가정함.
# 함수 내부를 단계별로 살펴보면,
# 먼저 scratchpad 변수를 빈 문자열로 초기화함.
# 이 변수는 에이전트의 사고 과정을 누적 저장하는 공간임. 매 반복마다 Thouth, Action, Action Input, Observation 이 추가됨.
# 반복문 안에서 _render_prompt 함수를 호출해 완성된 프롬프트를 생성하고, llm.invoke() 에 전달해 LLM의 응답을 받음.
# LLM 응답에서 content 속성을 추출해서 텍스트로 변환한 후, _parse_action_and_input 함수로 파싱함. 파싱 결과에 따라 네가지 경우로 분기함.
# 
# 첫째, tool이 None 인경우 LLM이 정해진 형식에 따르지 않았음을 의미함.
# 이 경우 형식을 준수하라는 힌트를 scratchpad 에 추가하고 다음 반복으로 의미함. 이 경우 최종 답변과 전체 실행 로그를 딕셔너리 형태로 반환하며 함수를 종료함
# 
# 둘째, tool이 "FINAL"인 경우는 에이전트가 최종 답변을 생성햇음을 의미함. 이 경우 최종 답변과 전체 실행로그를 딕셔너리 형태로 반환하며 함수를 종료함.
# 
# 셋째, tool이 tool_map에 없으면 LLM이 잘못된 도구이름을 출력한 것이므로 에러메시지를 Observation 으로 추가하고 다음 반복으로 넘김.
# 
# 넷째, 정상적인 경우에는 tool_map 에서 해당 도구를 찾아 invoke() 메서드로 실행함.
# 실행 결과를 _observation_to_text 함수로 문자열로 변환해 Observation 으로 scratchpad 에 추가함.
# 도구 실행 중 에러가 발생하면 에러 메시지를 Observation 으로 추가함.
# 
# 최대 반복 횟수에 도달하면 반복 초과 메시지와 함께 실행로그를 반환함. 이는 무한 루프를 방지하고, 복잡한 질문에 적절한 피드백을 제공하기 위함임.

# 이제 구현한 ReAct 에이전트를 실제로 테스트해 보겠음.
# 가장 먼저 '미국의 블록체인 시장 규모가 어떤가요?'라는 질문을 입력함. 이 질문은 usa_blockchain_trend_searcher 도구를 사용해야 해결할 수 있음.
"""
try : 
    q1 = "미국 블록체인 시장 발전 개요에 대해 정리해줘?"
    out1 = run_react(q1, max_iters = 8)
    print('최종 답변:', out1["output"])
    print("\n=== 실행로그(Thought/ Action/ Action Input/ Observation) ===\n")
    print(out1["log"])
except Exception as e:
    print(f"단건 질의 중 오류: {e}")
"""

# run_react 함수를 호출해 사용자의 질문을 에이전트에게 전달함
# 에이전트는 이 질문을 받으면 앞서 설정한 프롬프트에 따라 Thought->Action->Action Input-> Observation 사이클을 실행하게 됨.
# 반환된 결과에서 out1["output"]은 최종 답변을 out1["log"]는 에이전트 전체 사고 과정을 담고 있음.     

# react_template 변수를 살펴보면 마지막 부분이 {agent_scratchpad}로 마무리됨.
# 에이전트는 바로 그 위치에서 'Thought: ' 로 부터 글을 작성하기 시작함. 실행된 전체 과정을 단계별로 살펴보면 다음과 같음.
#   - Thought(첫번째 생각): 미국의 블록체인 시장 규모에 대한 최신 동향과 수치를 확인하기 위해 관련 도구를 사용해야 함.
#   - Action(선택된 도구): usa_blockchain_trend_searcher
#   - Action Input(입력된 검색어): "미국의 블록체인 시장 규모 및 성장 동향"
#   - Observation(도구 실행 결과): 2023년 기준 미국 블로체인 시장 규모는 100억 달러로 추정되며, 연평균 성장률 ...생략...

#   - Thought(두번째 생각): 이제 최종 답변을 알겠습니다.
#   - Observation(최종응답): 미국의 블록체인 시장 규모는 2023년 100억 달러로 추정되며, 연평균 성장률(CAGR)은 약 35%로 매우 빠르게 성장하고 있습니다. ...생략...

# 에이전트는  사용자 입력을 분석해 첫번재 생각(Thought)에서 미국의 블록체인 관련 도구를 사용해야 한다고 판단했음.
# 결과적으로 Action 부분에서 사용 가능한 세개의 도구 중에서 usa_blockchain_trend_searcher를 선택했고, Action Input 부분에 '미국의 블록체인 시장 규모 및 성장 동향'이라는 검색어를 입력해 검색을 수행함.
# Action 과 Action Input은 첫번째 Thought를 통해 결정됨.
# 검색을 통해 나온 결과에는 시장 규모의 구체적인 수치 데이터와 연평균 성장률, 그리고 금융, 공금망, 의료 등 관련 산업 분야의 동향 정보가 포함돼 있었음.
# 에이전트는 이러한 검색 결과를 확인한 후 두번째 생각(Thought)에서 질문을 해결하기에 충분한 정보를 확보했다고 판단했고, 이제 더이상 검색을 시도하지 않고 Final Answer 부분에서
# 최종 답변을 작성했음. 최종 답변은 out1['output']에 저장돼 있음.

# 이제 여러 도구를 활용하는 독립적인 질문을 테스트해 보겠음. 아래의 질문은 세개의 도구를 모두 사용해야만 답변 가능한 질문임.
try:
    q2 = "미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해주고 미국의 블록체인 스타트업 지원 제도를 정리해 주세요"
    out2 = run_react(q2, max_iters=10)
    print("최종 답변 : ", out2["output"])
    print("\n===실행 로그(Thought/ Action/ Action Input/ Observation) ===\n")
    print(out2["log"])
except Exception as e:
    print("멀티 쿼리 중 오류: {e}")


최종 답변 :  반복 한도를 초과합니다. 질문을 더 구체화해 주세요.

===실행 로그(Thought/ Action/ Action Input/ Observation) ===

Question: 미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해주고 미국의 블록체인 스타트업 지원 제도를 정리해 주세요

Thought: 먼저 미국 ICT 정책의 핵심 축과 추진체계를 확인하겠습니다.

Action: usa_ict_trend_searcher

Action Input: 미국의 주요 ICT 정책, 디지털 인프라, 반도체, AI, 사이버보안, 디지털정부 및 추진체계

Observation: 미국은 「Infrastructure Investment and Jobs Act」의 BEAD 프로그램을 통한 광대역망 확충, 「CHIPS and Science Act」에 따른 반도체 생산·연구개발 지원, NIST 중심의 AI 위험관리 및 표준화, 국가 사이버보안 전략, 연방정부 클라우드·데이터 현대화를 주요 축으로 추진한다. 연방정부가 전략과 재원을 제공하되 주정부, 개별 부처, 민간기업이 집행하는 분권형 구조가 강하다. AI와 디지털자산 정책은 행정부 교체에 따라 규제 방향이 달라질 수 있으며, 기술 경쟁력과 국가안보가 정책의 중요한 판단 기준이다.

Thought: 다음으로 일본 ICT 정책을 조사해 미국과 비교할 기준을 마련하겠습니다.

Action: japan_ict_trend_searcher

Action Input: 일본의 주요 ICT 정책, 디지털청, Society 5.0, 디지털정부, AI, Beyond 5G, 반도체 및 경제안보 정책

Observation: 일본은 Society 5.0을 장기 비전으로 삼고 디지털청을 중심으로 행정시스템 표준화, 마이넘버 활용, 정부 클라우드, 지역 디지털화 등을 추진한다. AI, 반도체, 데이터 유통, Beyond 5G·6G, 사이버보안, 경제안보도 핵심 분야다. 미국보다 중앙정부가 국가 전략과 민관 협력 프로젝트를 조정하는 성격